### Testing the pipeline:

There are 5 possible tests/queries in our catalog use-case:

1. ``Query category 1``: Family/category narrowing + Exact, keyword-heavy
2. ``Query category 2``: Range filtering or numeric constraint filtering 
3. ``Query category 3``: Compound structured query (i.e. multiple numeric constraints plus one exact match)
4. ``Query category 4``: Product Comparison / Recommendation
5. ``Query category 5``: Catalog Understanding / Descriptive Query

We are going to test every query type on our retrievers (vector-based, hybrid, hybrid+metadata filtering).

In [ ]:
# Query category 1
query_category1 = "gear with 22 teeth."
query_category1 = "all POM gears with module 2.0 and article number SH20110HF"
query_category1 = "Give me the entire row of spur gears made of polyacetal (POM) where ZZ=12"

# Query category 2
query_category2 = "give me POM gear rows between ZZ=12 and ZZ=14"
query_category2 = "give me POM gear rows with torque > 200Ncm and teeth count < 23 teeth"

# Query category 3
query_category3 = "Give me the entire row of spur gears made of polyacetal (POM) where ZZ=24 and ØKK=52."
query_category3 = "I need a module 2.0 spur gear with torque > 201 Ncm but in the same time teeth count < 23 teeths."

# Query category 4
query_category4 = "Which gear is better for compact high-torque use?"

# Query category 5
query_category5 = "What is the difference between spur gears and bevel gears?"

# Helpers
response_language_de = "Answer in German."
query_helper = ", then explain how you found the answer."

___
##### 📊 TESTING basic ``Vector- and Hybrid-retrieval``:

##### Display retrieved nodes

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

# retriever.retrieve(query) → returns retrieved nodes
vector_results_test1 = vector_retriever.retrieve(query)
hybrid_results_test1 = hybrid_retriever.retrieve(query)

print("\n--- BASIC VECTOR RETRIEVAL ---")
for r in vector_results_test1[:5]:
    print(r.score, r.node.node_id, r.node.metadata)

print("\n--- BASIC HYBRID RETRIEVAL ---")
for r in hybrid_results_test1[:5]:
    print(r.score, r.node.node_id, r.node.metadata)

##### Display LLM final answer

In [ ]:
# query_engine.query(query) → returns the synthesized LLM answe
vector_query_engine_test1 = RetrieverQueryEngine.from_args(retriever=vector_retriever)
hybrid_query_engine_test2 = RetrieverQueryEngine.from_args(retriever=hybrid_retriever)
vector_based_response = vector_query_engine_test1.query(query+response_language_de)
print(str(vector_based_response))
hybrid_based_response = hybrid_query_engine_test2.query(query+response_language_de)
print(str(hybrid_based_response))

___
##### 📊 TESTING basic ``Hybrid-retrieval with metadata filtering``:

##### Display retrieved nodes

In [ ]:
def retrieve(user_query: str, llm, use_hybrid: bool = False):
    intent = extract_query_intent(user_query, llm)
    print("-----------------intent------------")
    print(intent)
    retriever = build_filtered_retriever(intent, use_hybrid=use_hybrid)
    return retriever.retrieve(user_query)

# metadata filtering - vector-retriever
results1 = retrieve(
    user_query=query,
    llm=Settings.llm,
    use_hybrid=False,
)

# metadata filtering - hybrid-retriever
results2 = retrieve(
    user_query=query,
    llm=Settings.llm,
    use_hybrid=True,
)

print("-----------------results1------------")
for node in results1:
    # print(node.metadata)
    print(node.metadata.get("node_type"))
    print(node.metadata.get("family"))
    print(node.metadata.get("art_nr"))
    # print(node.text)
    print("---")

print("-----------------results2------------")
for node in results2:
    # print(node.metadata)
    print(node.metadata.get("node_type"))
    print(node.metadata.get("family"))
    print(node.metadata.get("art_nr"))
    # print(node.text)
    print("---")


##### Display LLM final answer

In [ ]:
intent = extract_query_intent(query, Settings.llm)
# print(intent)
metadata_retriever = build_filtered_retriever(intent, use_hybrid=True)

# query_engine.query(query) → returns the synthesized LLM answe
metadata_engine_test1 = RetrieverQueryEngine.from_args(retriever=metadata_retriever)
metadata_response = metadata_engine_test1.query(query)
print(str(metadata_response))